In [ ]:
import myFunctions as myF
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Set font configurations for Matplotlib (Times New Roman for English, SimHei for Chinese)
plt.rcParams['font.family'] = ['Times New Roman', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

# --- Load dataset ---

In [ ]:
# Note: Loading only the first 88,000 rows as per original script
df = pd.read_csv('./merged_combined_data.csv').iloc[:88000]

# Feature Engineering: Extract Month and Day
df['转化月份和日'] = pd.to_datetime(df['日期'], format='ISO8601')
df['月份'] = df['转化月份和日'].dt.month
df['日'] = df['转化月份和日'].dt.day
df.drop(columns=['转化月份和日'], inplace=True)

# Visualization: Price Distribution 
plt.figure(figsize=(10, 6), dpi=400)
sns.histplot(df['实时出清价格(元/MWh)'], kde=True, bins='auto', stat='probability',
             color=plt.cm.Blues(0.5))
# plt.title('Distribution of "Real-time Clearing Price (CNY/MWh)" in Matching Data Set', fontsize=14)
plt.xlabel('Real-time clearing price (CNY/MWh)', fontsize=24)
plt.ylabel('Proportion', fontsize=24)
plt.xlim(-20, 1520)
plt.xticks([0, 200, 400, 600, 900, 1200, 1500])
# Adjust tick label size
plt.tick_params(axis='both', which='major', labelsize=24)
plt.tight_layout()
plt.show()

# --- Cluster Analysis ---

In [ ]:
# Define columns for clustering
columns = ['序号', '非市场化机组出力(MW)', '水电实际值(MW)', '风电实际值(MW)', '光伏实际值(MW)',
           '新能源总加实际值(MW)', '实时出清价格(元/MWh)', '日前出清价格(元/MWh)', '检修容量(MW)',
           '日前联络线电力值(MW)', '省内负荷预测电力值(MW)', '日前联络线和省内负荷预测总加电力值(MW)',
           '统调用电负荷电力值(MW)', '新能源出力预测电力值(MW)', '新能源出力预测风电(MW)', '新能源出力预测光伏(MW)',
           't2m', 'u10', 'v10', 'tp', '月份', '日']

X = df[columns].copy()
X = X.fillna(X.mean())

# Determine optimal k using Silhouette Score
best_k = myF.ClusterAnalysis().get_k_cluster(X.iloc[4:], k_max=50, title=False)

# Perform K-Means clustering with optimal k
Y = X.iloc[4:].copy()
# scaler = StandardScaler()
# Y_scaled = scaler.fit_transform(Y)
print(f"Data shape for clustering: {Y.shape}")
kmeans = KMeans(n_clusters=best_k, random_state=10, n_init=10)
# kmeans.fit(Y_scaled)
kmeans.fit(Y)
cluster_labels = kmeans.labels_

# Save cluster labels to dataframe
Y['cluster'] = cluster_labels

# Calculate statistics (Mean and Std) for each cluster
mean_df = Y.groupby('cluster').mean()
std_df = Y.groupby('cluster').std()

# Calculate proportion of each cluster
cluster_counts = Y['cluster'].value_counts(normalize=True)
cluster_proportions = cluster_counts * 100 

# Display settings
pd.set_option('display.max_columns', None)  # Force display of all columns
print(std_df)

# --- LSTM Forecasting Experiments ---

In [ ]:
# Define base columns for LSTM models
columns = ['序号', '非市场化机组出力(MW)', '水电实际值(MW)', '风电实际值(MW)', '光伏实际值(MW)',
           '新能源总加实际值(MW)', '实时出清价格(元/MWh)', '日前出清价格(元/MWh)', '检修容量(MW)',
           '日前联络线电力值(MW)', '省内负荷预测电力值(MW)', '日前联络线和省内负荷预测总加电力值(MW)',
           '统调用电负荷电力值(MW)', '新能源出力预测电力值(MW)', '新能源出力预测风电(MW)', '新能源出力预测光伏(MW)',
           't2m', 'u10', 'v10', 'tp', '月份', '日']

## Experiment 1: Basic LSTM Usage

In [ ]:
data = df[columns].copy()
print(f"Basic LSTM Data Shape: {data.shape}")

# Preprocess time column
df['时点'] = df['时点'].replace('24:00', '23:59')
data['_date'] = pd.to_datetime(df['日期'] + ' ' + df['时点'], format='ISO8601')

# Run LSTM
result_json = myF.LSTMAnalysis().run_time(
    data_with_time=data.copy().dropna(), 
    time_name='_date', 
    name='实时出清价格(元/MWh)',
    seq_len=1*96, 
    batch_size=2048, 
    epochs=50, 
    loss_method='mean_absolute_error',
    test_column='_date', 
    test_range=['2024-04-01', '2024-06-01']
)

# Save results
output_file = f'./results/LSTM_results-20240401.json'
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(result_json)
print(f"Results written to {output_file}")


## Experiment 2: LSTM with Category (Clustering)

In [ ]:
data = df[columns].copy().dropna()

# Perform clustering on input data to create a 'category' feature
kmeans = KMeans(n_clusters=2, random_state=10, n_init=10) 
kmeans.fit(data)
data['category'] = kmeans.labels_

# Preprocess time
df['时点'] = df['时点'].replace('24:00', '23:59')
data['_date'] = pd.to_datetime(df['日期'] + ' ' + df['时点'], format='ISO8601')

# Run LSTM with categorical column
result_json = myF.LSTMAnalysis().run_time(
    data_with_time=data.copy().dropna(), 
    time_name='_date', 
    name='实时出清价格(元/MWh)',
    seq_len=1*96, 
    categorical_columns=['category'], 
    batch_size=2048, 
    epochs=50, 
    loss_method='mean_absolute_error',
    test_column='_date', 
    test_range=['2024-04-01', '2024-06-01']
)

# Save results
output_file = f'./results/LSTM_results-20240401-category.json'
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(result_json)
print(f"Results written to {output_file}")

## Experiment 3: LSTM with Feature Construction (FC)

In [ ]:
data = df[columns].copy()
print(f"FC Data Shape (Before): {data.shape}")

# Create lag features (shifting columns)
shift_columns = ['风电实际值(MW)', '光伏实际值(MW)', '日前出清价格(元/MWh)',
                 '省内负荷预测电力值(MW)', '统调用电负荷电力值(MW)',
                 '实时出清价格(元/MWh)',  '新能源出力预测电力值(MW)']
for col in shift_columns:
    for i in range(1, 5):
        data[f'{col}_{i}'] = data[f'{col}'].shift(i) 

print(f"FC Data Shape (After): {data.shape}")

# Preprocess time
df['时点'] = df['时点'].replace('24:00', '23:59')
data['_date'] = pd.to_datetime(df['日期'] + ' ' + df['时点'], format='ISO8601')

# Run LSTM
result_json = myF.LSTMAnalysis().run_time(
    data_with_time=data.copy().dropna(), 
    time_name='_date', 
    name='实时出清价格(元/MWh)',
    seq_len=1*96, 
    batch_size=1024, 
    epochs=50, 
    loss_method='mean_absolute_error',
    test_column='_date', 
    test_range=['2024-04-01', '2024-06-01']
)

# Save results
output_file = f'./results/LSTM_results-20240401-FC.json'
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(result_json)
print(f"Results written to {output_file}")

## Experiment 4: LSTM with Feature Selection (FS)

In [ ]:
data = df[columns].copy()

# Perform feature selection using Symmetric Uncertainty
selected_features = myF.SymmetricUncertainty().select_features(
    data=data.dropna(), 
    target_col='实时出清价格(元/MWh)'
)
print(f'Selected features: {selected_features}')

# Ensure target column is included
selected_features.append('实时出清价格(元/MWh)')
data = data[selected_features]
print(f"FS Data Shape: {data.shape}")

# Preprocess time
df['时点'] = df['时点'].replace('24:00', '23:59')
data['_date'] = pd.to_datetime(df['日期'] + ' ' + df['时点'], format='ISO8601')

# Run LSTM
result_json = myF.LSTMAnalysis().run_time(
    data_with_time=data.copy().dropna(), 
    time_name='_date', 
    name='实时出清价格(元/MWh)',
    seq_len=1*96, 
    batch_size=2048, 
    epochs=50, 
    loss_method='mean_absolute_error',
    test_column='_date', 
    test_range=['2024-04-01', '2024-06-01']
)

# Save results
output_file = f'./results/LSTM_results-20240401-FS.json'
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(result_json)
print(f"Results written to {output_file}")

## Experiment 5: LSTM with FC and FS

In [ ]:
data = df[columns].copy()

# 1. Feature Construction (FC)
shift_columns = ['风电实际值(MW)', '光伏实际值(MW)', '日前出清价格(元/MWh)',
                 '省内负荷预测电力值(MW)', '统调用电负荷电力值(MW)',
                 '实时出清价格(元/MWh)',  '新能源出力预测电力值(MW)']
for col in shift_columns:
    for i in range(1, 5):
        data[f'{col}_{i}'] = data[f'{col}'].shift(i) 

# 2. Feature Selection (FS)
selected_features = myF.SymmetricUncertainty().select_features(
    data=data.dropna(), 
    target_col='实时出清价格(元/MWh)'
)
print(f'Selected features: {selected_features}')

selected_features.append('实时出清价格(元/MWh)')
data = data[selected_features]
print(f"FC-FS Data Shape: {data.shape}")

# Preprocess time
df['时点'] = df['时点'].replace('24:00', '23:59')
data['_date'] = pd.to_datetime(df['日期'] + ' ' + df['时点'], format='ISO8601')

# Run LSTM
result_json = myF.LSTMAnalysis().run_time(
    data_with_time=data.copy().dropna(), 
    time_name='_date', 
    name='实时出清价格(元/MWh)',
    seq_len=1*96, 
    batch_size=2048, 
    epochs=50, 
    loss_method='mean_absolute_error',
    test_column='_date', 
    test_range=['2024-04-01', '2024-06-01']
)

# Save results
output_file = f'./results/LSTM_results-20240401-FC-FS.json'
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(result_json)
print(f"Results written to {output_file}")

## Experiment 6: LSTM with FS and VMD (Variational Mode Decomposition)

In [ ]:
data = df[columns].copy()

# Feature Selection
selected_features = myF.SymmetricUncertainty().select_features(
    data=data.dropna(), 
    target_col='实时出清价格(元/MWh)'
)
print(f'Selected features: {selected_features}')

selected_features.append('实时出清价格(元/MWh)')
data = data[selected_features]

# Preprocess time
df['时点'] = df['时点'].replace('24:00', '23:59')
data['_date'] = pd.to_datetime(df['日期'] + ' ' + df['时点'], format='ISO8601')

# Run LSTM with VMD
result_json = myF.LSTMAnalysis().run_time(
    data_with_time=data.copy().dropna(), 
    time_name='_date', 
    name='实时出清价格(元/MWh)',
    seq_len=1*96, 
    batch_size=2048, 
    epochs=50, 
    loss_method='mean_absolute_error',
    test_column='_date', 
    test_range=['2024-04-01', '2024-06-01'],
    vmd=True
)

# Save results
output_file = f'./results/LSTM_results-20240401-FS-VMD.json'
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(result_json)
print(f"Results written to {output_file}")

## Experiment 7: LSTM with FC, FS, and VMD

In [ ]:
data = df[columns].copy()

# 1. Feature Construction (FC)
shift_columns = ['风电实际值(MW)', '光伏实际值(MW)', '日前出清价格(元/MWh)',
                 '省内负荷预测电力值(MW)', '统调用电负荷电力值(MW)',
                 '实时出清价格(元/MWh)',  '新能源出力预测电力值(MW)']
for col in shift_columns:
    for i in range(1, 5):
        data[f'{col}_{i}'] = data[f'{col}'].shift(i) 

# 2. Feature Selection (FS)
# Note: Using small 's' symmetric_uncertainty() based on original code, 
# ensuring class naming consistency with your module import.
selected_features = myF.SymmetricUncertainty().select_features(
    data=data.dropna(), 
    target_col='实时出清价格(元/MWh)'
)
print(f'Selected features: {selected_features}')

selected_features.append('实时出清价格(元/MWh)')
data = data[selected_features]
print(f"FC-FS-VMD Data Shape: {data.shape}")

# Preprocess time
df['时点'] = df['时点'].replace('24:00', '23:59')
data['_date'] = pd.to_datetime(df['日期'] + ' ' + df['时点'], format='ISO8601')

# Run LSTM with VMD
result_json = myF.LSTMAnalysis().run_time(
    data_with_time=data.copy().dropna(), 
    time_name='_date', 
    name='实时出清价格(元/MWh)',
    seq_len=1*96, 
    batch_size=2048, 
    epochs=50, 
    loss_method='mean_absolute_error',
    test_column='_date', 
    test_range=['2024-04-01', '2024-06-01'],
    vmd=True
)

# Save results
output_file = f'./results/LSTM_results-20240401-FC-FS-VMD.json'
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(result_json)
print(f"Results written to {output_file}")

## Experiment 8: LSTM with FS and EMD (Empirical Mode Decomposition)

In [ ]:
data = df[columns].copy()

# Feature Selection
selected_features = myF.SymmetricUncertainty().select_features(
    data=data.dropna(), 
    target_col='实时出清价格(元/MWh)'
)
print(f'Selected features: {selected_features}')

selected_features.append('实时出清价格(元/MWh)')
data = data[selected_features]

# Preprocess time
df['时点'] = df['时点'].replace('24:00', '23:59')
data['_date'] = pd.to_datetime(df['日期'] + ' ' + df['时点'], format='ISO8601')

# Run LSTM with EMD
result_json = myF.LSTMAnalysis().run_time(
    data_with_time=data.copy().dropna(), 
    time_name='_date', 
    name='实时出清价格(元/MWh)',
    seq_len=1*96, 
    batch_size=2048, 
    epochs=50, 
    loss_method='mean_absolute_error',
    test_column='_date', 
    test_range=['2024-04-01', '2024-06-01'],
    emd=True
)

# Save results
output_file = f'./results/LSTM_results-20240401-FS-EMD.json'
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(result_json)
print(f"Results written to {output_file}")

## Experiment 9: LSTM with FC, FS, and EMD

In [ ]:
data = df[columns].copy()

# 1. Feature Construction
shift_columns = ['风电实际值(MW)', '光伏实际值(MW)', '日前出清价格(元/MWh)',
                 '省内负荷预测电力值(MW)', '统调用电负荷电力值(MW)',
                 '实时出清价格(元/MWh)',  '新能源出力预测电力值(MW)']
for col in shift_columns:
    for i in range(1, 5):
        data[f'{col}_{i}'] = data[f'{col}'].shift(i)

# 2. Feature Selection
selected_features = myF.SymmetricUncertainty().select_features(
    data=data.dropna(), 
    target_col='实时出清价格(元/MWh)'
)
print(f'Selected features: {selected_features}')

selected_features.append('实时出清价格(元/MWh)')
data = data[selected_features]
print(f"FC-FS-EMD Data Shape: {data.shape}")

# Preprocess time
df['时点'] = df['时点'].replace('24:00', '23:59')
data['_date'] = pd.to_datetime(df['日期'] + ' ' + df['时点'], format='ISO8601')

# Run LSTM with EMD
result_json = myF.LSTMAnalysis().run_time(
    data_with_time=data.copy().dropna(), 
    time_name='_date', 
    name='实时出清价格(元/MWh)',
    seq_len=1*96, 
    batch_size=2048, 
    epochs=50, 
    loss_method='mean_absolute_error',
    test_column='_date', 
    test_range=['2024-04-01', '2024-06-01'],
    emd=True
)

# Save results
output_file = f'./results/LSTM_results-20240401-FC-FS-EMD.json'
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(result_json)
print(f"Results written to {output_file}")

## Experiment 10: LSTM with FS and CEEMDAN

In [ ]:
data = df[columns].copy()

# Feature Selection
selected_features = myF.SymmetricUncertainty().select_features(
    data=data.dropna(), 
    target_col='实时出清价格(元/MWh)'
)
print(f'Selected features: {selected_features}')

selected_features.append('实时出清价格(元/MWh)')
data = data[selected_features]

# Preprocess time
df['时点'] = df['时点'].replace('24:00', '23:59')
data['_date'] = pd.to_datetime(df['日期'] + ' ' + df['时点'], format='ISO8601')

# Run LSTM with CEEMDAN
result_json = myF.LSTMAnalysis().run_time(
    data_with_time=data.copy().dropna(), 
    time_name='_date', 
    name='实时出清价格(元/MWh)',
    seq_len=1*96, 
    batch_size=2048, 
    epochs=50, 
    loss_method='mean_absolute_error',
    test_column='_date', 
    test_range=['2024-04-01', '2024-06-01'],
    ceemdan=True
)

# Save results
output_file = f'./results/LSTM_results-20240401-FS-CEEMDAN.json'
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(result_json)
print(f"Results written to {output_file}")

## Experiment 11: LSTM with FC, FS and CEEMDAN

In [ ]:
data = df[columns].copy()

# 1. Feature Construction
shift_columns = ['风电实际值(MW)', '光伏实际值(MW)', '日前出清价格(元/MWh)',
                 '省内负荷预测电力值(MW)', '统调用电负荷电力值(MW)',
                 '实时出清价格(元/MWh)',  '新能源出力预测电力值(MW)']
for col in shift_columns:
    for i in range(1, 5):
        data[f'{col}_{i}'] = data[f'{col}'].shift(i)

# 2. Feature Selection
selected_features = myF.SymmetricUncertainty().select_features(
    data=data.dropna(), 
    target_col='实时出清价格(元/MWh)'
)
print(f'Selected features: {selected_features}')

selected_features.append('实时出清价格(元/MWh)')
data = data[selected_features]
print(f"FC-FS-EMD Data Shape: {data.shape}")

# Preprocess time
df['时点'] = df['时点'].replace('24:00', '23:59')
data['_date'] = pd.to_datetime(df['日期'] + ' ' + df['时点'], format='ISO8601')

# Run LSTM with CEEMDAN
result_json = myF.LSTMAnalysis().run_time(
    data_with_time=data.copy().dropna(), 
    time_name='_date', 
    name='实时出清价格(元/MWh)',
    seq_len=1*96, 
    batch_size=2048, 
    epochs=50, 
    loss_method='mean_absolute_error',
    test_column='_date', 
    test_range=['2024-04-01', '2024-06-01'],
    ceemdan=True
)

# Save results
output_file = f'./results/LSTM_results-20240401-FC-FS-CEEMDAN.json'
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(result_json)
print(f"Results written to {output_file}")

# --- Pattern Matching for Interval Feature Extraction ---

In [ ]:
# Uses Cosine Similarity to find similar historical moments and analyze price distributions.
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import json

olumns = ['序号', '非市场化机组出力(MW)', '水电实际值(MW)', '风电实际值(MW)', '光伏实际值(MW)',
       '新能源总加实际值(MW)', '实时出清价格(元/MWh)', '日前出清价格(元/MWh)', '检修容量(MW)',
       '日前联络线电力值(MW)', '省内负荷预测电力值(MW)', '日前联络线和省内负荷预测总加电力值(MW)',
       '统调用电负荷电力值(MW)', '新能源出力预测电力值(MW)', '新能源出力预测风电(MW)', '新能源出力预测光伏(MW)',
       't2m', 'u10', 'v10', 'tp', '月份', '日']

data = df[columns].copy()
df['时点'] = df['时点'].replace('24:00', '23:59')
data['_date'] = pd.to_datetime(df['日期'] + ' ' + df['时点'], format='ISO8601')

# Create lagged features for similarity matching (t-1)
for name in ['风电实际值(MW)', '光伏实际值(MW)', '日前出清价格(元/MWh)',
             '省内负荷预测电力值(MW)', '统调用电负荷电力值(MW)']:
    data[name] = data[name].shift(1)

# Create lagged price features
for i in range(1, 5):
    data[f'price_{i}'] = data['实时出清价格(元/MWh)'].shift(i)

# Define feature set for calculating Cosine Similarity
features_for_cosine = ['风电实际值(MW)', '光伏实际值(MW)',
                       '省内负荷预测电力值(MW)', '统调用电负荷电力值(MW)',
                       '日前出清价格(元/MWh)',
                       't2m', 'u10', 'v10', 'tp', '序号', '日', '月份',
                       'price_1', 'price_2', 'price_3', 'price_4']
data.dropna(inplace=True)

# Define test range
test_range = ['2024-04-01', '2024-06-01']
test_mask = (data['_date'] >= pd.to_datetime(test_range[0])) & (data['_date'] <= pd.to_datetime(test_range[1]))
test_data = data[test_mask]

# Parameters for matching
n = 1000                # Max number of similar points to consider
threshold = 0.99        # Cosine similarity threshold
threshold_percentage = 0.11 # Threshold for filtering low-frequency bins
num_bins = 10           # Number of bins for price distribution

test_results = []

# Iterate through each test data point
for index, test_row in tqdm(test_data.iterrows(), total=test_data.shape[0]):
    test_time = test_row['_date']
    
    # Select only historical data (prior to current test time) as candidates
    candidate_data = data[data['_date'] < test_time]
    candidate_data_cosine = candidate_data[features_for_cosine].values
    
    # Get features for current test point
    test_features = test_row[features_for_cosine].values.reshape(1, -1)
    
    # Calculate Cosine Similarity
    similarities = cosine_similarity(test_features, candidate_data_cosine)[0]
    
    # Filter candidates based on threshold
    valid_indices = np.where(similarities >= threshold)[0]
    filtered_similarities = similarities[valid_indices]
    
    # Skip if no matches found
    if len(valid_indices) == 0:
        continue
    
    # Sort by similarity and take top n
    top_n_indices = valid_indices[np.argsort(filtered_similarities)[-n:][::-1]]
    
    # Extract prices of the top matched points
    top_n_prices = candidate_data.iloc[top_n_indices]['实时出清价格(元/MWh)']
    
    # Discretize prices into bins (Quantile cut)
    price_bins = pd.qcut(top_n_prices, q=num_bins, duplicates='drop')
    
    # Get bin boundaries
    price_bins_boundaries = [interval.left for interval in price_bins.cat.categories]
    if len(price_bins.cat.categories) > 0:
        price_bins_boundaries.append(price_bins.cat.categories[-1].right)

    # Calculate frequency proportion per bin
    frequency_table = price_bins.value_counts(normalize=True)
    
    # Filter out bins with proportion below threshold
    valid_bins = frequency_table[frequency_table >= threshold_percentage].index
    if not valid_bins.empty:
        top_n_prices = top_n_prices[price_bins.isin(valid_bins)]

    # Calculate statistical metrics for the filtered distribution
    mean_price = top_n_prices.mean()
    std_price = top_n_prices.std()
    median_price = top_n_prices.median()
    quantile_25 = top_n_prices.quantile(0.25)
    quantile_75 = top_n_prices.quantile(0.75)

    # Store results
    test_results.append({
        'price': test_row['实时出清价格(元/MWh)'],
        'test_index': index,
        'test_time': str(test_time),
        'mean_price': mean_price,
        'std_price': std_price,
        'median_price': median_price,
        '25th_quantile': quantile_25,
        '75th_quantile': quantile_75,
        'matching_size': len(top_n_prices),
        'price_bins': price_bins_boundaries
    })

# Save matching results to JSON
output_file = f'./results/interval.json'
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(test_results, f, ensure_ascii=False, indent=4)

print(f"Results saved to {output_file}")